In [12]:
import os
import win32com.client
import pandas as pd
from glob import glob
import tabula
from thefuzz import process

In [ ]:
def process_doc_file(file_path):
    all_data = []
    try:
        word = win32com.client.Dispatch("Word.Application")
        try:
            doc = word.Documents.Open(file_path)
        except Exception as e:
            print(f"无法打开Word文件: {file_path}，已跳过。错误信息: {str(e)}")
            word.Quit()
            return all_data  # 跳过无法打开的文件

        for table_idx, table in enumerate(doc.Tables, 1):
            try:
                row_count = table.Rows.Count
                col_count = table.Columns.Count
                print(f"表格 {table_idx}: {row_count}行 × {col_count}列")
                for row in range(1, row_count + 1):
                    row_data = []
                    for col in range(1, col_count + 1):
                        try:
                            cell_text = table.Cell(row, col).Range.Text
                            row_data.append(cell_text.strip().replace('\r\x07', ''))
                        except:
                            row_data.append("ERROR_CELL")
                    # 如果第四列存在且包含‘车’，则跳过该行
                    if len(row_data) >= 4 and '车' in row_data[3]:
                        continue
                    all_data.append(row_data)
            except Exception as e:
                print(f"跳过表格 {table_idx}: {str(e)}")
        doc.Close(False)
        word.Quit()
    except Exception as e:
        print(f"Word进程启动失败或关闭失败: {str(e)}")
    return all_data

def process_pdf_file(file_path):
    all_data = []
    try:
        # 读取所有表格
        tables = tabula.read_pdf(file_path, pages='all', multiple_tables=True, lattice=True)
        for table_idx, df in enumerate(tables, 1):
            print(f"PDF表格 {table_idx}: {df.shape[0]}行 × {df.shape[1]}列")
            for row in df.itertuples(index=False):
                row_data = list(row)
                # 如果第四列存在且包含‘车’，则跳过该行
                if len(row_data) >= 4 and '车' in str(row_data[3]):
                    continue
                all_data.append(row_data)
    except Exception as e:
        print(f"PDF文件打开失败: {str(e)}")
    return all_data

# 主程序
folder_path = r"C:\Users\Lenovo\Desktop\Dissertaion\China Data\Range"
output_path = os.path.join(folder_path, "output.xlsx")

doc_files = glob(os.path.join(folder_path, "*.doc"))
pdf_files = glob(os.path.join(folder_path, "*.pdf"))

all_data = []

# 处理doc文件
if doc_files:
    for file in doc_files:
        print(f"\n处理Word文件: {os.path.basename(file)}")
        all_data.extend(process_doc_file(file))

# 处理pdf文件
if pdf_files:
    for file in pdf_files:
        print(f"\n处理PDF文件: {os.path.basename(file)}")
        all_data.extend(process_pdf_file(file))


if all_data:
    # Remove illegal characters from all cells
    def clean_cell(cell):
        if isinstance(cell, str):
            return cell.replace('\x07', '')
        return cell
    cleaned_data = [[clean_cell(cell) for cell in row] for row in all_data]
    pd.DataFrame(cleaned_data).to_excel(output_path, index=False)
    print(f"\n成功保存到: {output_path}")
else:
    print("未提取到有效数据！")



处理Word文件: 2056aa38a0224740b11c24934817592f.doc
表格 1: 84行 × 9列
表格 2: 31行 × 9列
表格 3: 49行 × 9列
表格 4: 175行 × 9列


In [29]:
################ Import Range Data
range_df = pd.read_excel(r"C:\Users\Lenovo\Desktop\Dissertaion\China Data\Range\output.xlsx")
range_df.head()

,0,1,2,3,4,5,6,7,8,9,10
0,序号,汽车生产企业名称,车辆型号,通用名称,纯电动续驶里程(km),整车整备质量(kg),动力蓄电池组总质量(kg),动力蓄电池组总能量（kWh）,备注,NaN,NaN
1,NaN,宝马（中国）汽车贸易有限公司,i7 M70L xDrive 81EH,创新纯电动BMW i7 M70L,610 （CLTC）,2780,684.2,105.7,NaN,NaN,NaN
2,NaN,中国第一汽车集团有限公司,CA7000BEVC,奔腾小马,122,685,86±3,9.4,NaN,NaN,NaN
3,NaN,ERROR_CELL,CA7000BEVB,奔腾小马,170,715,124±3,13.9,NaN,NaN,NaN
4,NaN,ERROR_CELL,CA7000BEVA,奔腾小马,122,685,90±2,9.98,NaN,NaN,NaN


In [30]:
################ Process range data

# Set the first row as column names and remove it from data
range_df.columns = range_df.iloc[0]
range_df = range_df[1:].reset_index(drop=True)

# Only keep columns 2 and 4 to 8
range_df = range_df.iloc[:, [1, 3, 4, 5, 6, 7]]

# Rename columns for clarity
range_df.columns = ['firm', 'model', 'range', 'mass', 'batter_weight', 'battery_capacity']

# remove non-numeric characters from 'range' and keep only the number
range_df['range'] = range_df['range'].astype(str).str.extract(r'(\d+\.?\d*)')[0]

# remove '±x' from batter_weight and keep only the number
range_df['batter_weight'] = range_df['batter_weight'].astype(str).str.replace(r'±\d+\.?\d*', '', regex=True).str.extract(r'(\d+\.?\d*)')[0]

# Convert columns to numeric, errors='coerce' will turn non-convertible values to NaN
for col in ['range', 'mass', 'batter_weight', 'battery_capacity']:
    range_df[col] = pd.to_numeric(range_df[col], errors='coerce')

# For duplicate models, take firm name
def get_valid_firm(x):
    valid_firms = x[x != "ERROR_CELL"]
    if len(valid_firms) == 0:
        return None
    return valid_firms.mode().iloc[0] 

range_df = range_df.groupby('model').agg({
    'firm': get_valid_firm,
    'range': 'mean',
    'mass': 'mean',
    'batter_weight': 'mean',
    'battery_capacity': 'mean'
}).reset_index()


In [32]:
################ Import sales data
df = pd.read_excel('final_cleaned_data.xlsx')

In [33]:
################ Name matching using fuzzywuzzy
# Threshold 60 and 80
threshold_60 = 60
not_found_60 = []
matches_60 = {}

for m in df['model'].unique():
    match = process.extractOne(m, range_df['model'].unique())
    if match is None or match[1] < threshold_60:
        not_found_60.append(m)
    else:
        matches_60[m] = match

threshold_80 = 80
not_found_80 = []
matches_80 = {}

for m in df['model'].unique():
    match = process.extractOne(m, range_df['model'].unique())
    if match is None or match[1] < threshold_80:
        not_found_80.append(m)
    else:
        matches_80[m] = match

# Match results
diff = set(not_found_80) - set(not_found_60)
print(f"60分能匹配但80分不能匹配的model数量: {len(diff)}")
for m in list(diff)[:10]:  # 只展示前10个
    print(f"Model: {m}")
    print(f"Best match in range_df: {matches_60.get(m)}")

    # 找出60能匹配但80不能匹配的model，并展示它们在range_df中的最佳匹配对象和分数
diff = set(not_found_80) - set(not_found_60)
print(f"60分能匹配但80分不能匹配的model数量: {len(diff)}")
for m in list(diff)[:10]:  # 只展示前10个
    match_name, score = matches_60[m]
    print(f"Model: {m}")
    print(f"Best match in range_df: {match_name} (score: {score})")

    # 展示80分及以上能匹配到的model及其匹配对象和分数
print(f"80分及以上能匹配到的model数量: {len(matches_80)}")
for m in list(matches_80.keys())[:10]:  # 只展示前10个
    match_name, score = matches_80[m]
    print(f"Model: {m}")
    print(f"Best match in range_df: {match_name} (score: {score})")

60分能匹配但80分不能匹配的model数量: 186
Model: 名图
Best match in range_df: ('名爵eHS', 60)
Model: 凯翼昆仑
Best match in range_df: ('凯翼E5EV', 60)
Model: 翼虎
Best match in range_df: ('斯威大虎', 60)
Model: 五菱之光
Best match in range_df: ('五菱星光', 75)
Model: EVOS
Best match in range_df: ('新特AEVs', 77)
Model: 曹操60
Best match in range_df: ('06', 60)
Model: 俊风
Best match in range_df: ('风云 A8,ARRIZO 8', 60)
Model: 名爵ZS
Best match in range_df: ('名爵eHS', 67)
Model: 奥迪Q5L Sportback
Best match in range_df: ('K3', 60)
Model: 海马8S
Best match in range_df: ('S3', 60)
60分能匹配但80分不能匹配的model数量: 186
Model: 名图
Best match in range_df: 名爵eHS (score: 60)
Model: 凯翼昆仑
Best match in range_df: 凯翼E5EV (score: 60)
Model: 翼虎
Best match in range_df: 斯威大虎 (score: 60)
Model: 五菱之光
Best match in range_df: 五菱星光 (score: 75)
Model: EVOS
Best match in range_df: 新特AEVs (score: 77)
Model: 曹操60
Best match in range_df: 06 (score: 60)
Model: 俊风
Best match in range_df: 风云 A8,ARRIZO 8 (score: 60)
Model: 名爵ZS
Best match in range_df: 名爵eHS (score: 67)
Model: 

In [34]:
################ Merge range data into the main data and export
matched_models = [m for m, (match_name, score) in matches_80.items() if score >= 80]

filtered_df = df[df['model'].isin(matched_models)].copy()

range_df_merge = range_df.drop(columns=['firm', 'mass'])

model_to_best_match = {m: match_name for m, (match_name, score) in matches_80.items() if score >= 80}

filtered_df['model_matched'] = filtered_df['model'].map(model_to_best_match)

merged_df = pd.merge(
    filtered_df,
    range_df_merge,
    left_on='model_matched',
    right_on='model',
    how='left',
    suffixes=('', '_range')
)

merged_df = merged_df.drop(columns=['model_matched', 'model_range'])

print(merged_df[['model', 'range', 'battery_capacity']].head())

   model  range  battery_capacity
0  景逸S50  170.0             13.92
1    卡罗拉   55.0               NaN
2   云度π1  251.0             39.00
3   云度π3  315.0             47.00
4  传祺GS4   56.5               NaN


In [36]:
################ Export merged data
output_file = 'merged_data_with_range.xlsx'
merged_df.to_excel(output_file, index=False)